In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
from datetime import datetime, timedelta
import hashlib

# Configuration
TARGET = "machine_health.prod.sensor_readings_raw"
INTERVAL = 5

sensors = [
    ("MOTOR-001", "Motor"),
    ("MOTOR-002", "Motor"),
    ("COMP-001", "Compressor"),
    ("COMP-002", "Compressor")
]

axes = ["X", "Y", "Z"]


# Deterministic noise
def noise(key, low, high):
    value = int(hashlib.md5(key.encode()).hexdigest()[:8], 16) / 0xFFFFFFFF
    return low + (high - low) * value


# Generate sensor readings
def generate_reading(sensor_id, machine_type, axis, timestamp):
    bucket = int(timestamp.timestamp() // (INTERVAL * 60))
    key = f"{sensor_id}|{machine_type}|{axis}|{bucket}"

    load = 1.0 if 8 <= timestamp.hour < 18 else (
        0.75 if 6 <= timestamp.hour < 8 or 18 <= timestamp.hour < 21 else 0.55
    )

    vibration_base, temp_base, accel_base = (
        (0.18, 60.0, 0.20)
        if machine_type == "Motor"
        else (0.22, 65.0, 0.25)
    )

    axis_vibration = {"X": 0.00, "Y": 0.03, "Z": 0.06}[axis]
    axis_accel = {"X": 0.00, "Y": 0.03, "Z": 0.05}[axis]

    degradation = 0.25 if sensor_id in ["MOTOR-002", "COMP-002"] else 0.0

    vibration = (
        vibration_base + axis_vibration + load * 0.08 + degradation
        + noise(key + "|vibration", -0.03, 0.03)
    )

    temperature = (
        temp_base + load * 5 + degradation * 20
        + noise(key + "|temperature", -1.5, 1.5)
    )

    acceleration = (
        accel_base + axis_accel + vibration * 0.20 + degradation * 0.05
        + noise(key + "|acceleration", -0.03, 0.03)
    )

    return (
        round(max(0.05, acceleration), 4),
        round(max(0.05, vibration), 4),
        round(max(20.0, temperature), 2)
    )


# Get latest reading timestamp
latest = spark.table(TARGET).agg(
    F.max("reading_time").alias("latest")
).first()["latest"]

now = datetime.utcnow().replace(second=0, microsecond=0)

reading_time = latest + timedelta(minutes=INTERVAL) if latest else now - timedelta(minutes=INTERVAL)

# Generate and insert readings
if reading_time <= now:
    records = []

    for sensor_id, machine_type in sensors:
        for axis in axes:
            accel, vibration, temperature = generate_reading(
                sensor_id, machine_type, axis, reading_time
            )

            records.append((
                sensor_id, machine_type, reading_time, axis,
                accel, vibration, temperature
            ))

    schema = StructType([
        StructField("sensor_id", StringType(), False),
        StructField("machine_type", StringType(), False),
        StructField("reading_time", TimestampType(), False),
        StructField("axis", StringType(), False),
        StructField("peak_accel_g", DoubleType(), False),
        StructField("vibration_rms_mm_s", DoubleType(), False),
        StructField("temp_c", DoubleType(), False)
    ])

    df = spark.createDataFrame(records, schema)
    df.write.mode("append").saveAsTable(TARGET)

    print(f"Inserted {len(records)} readings at {reading_time}")
    display(df.orderBy("sensor_id", "axis"))

else:
    print(f"No new readings required. Latest timestamp: {latest}")

/home/spark-ed58cd1f-d0a0-4ab7-b235-38/.ipykernel/62/command-6922329854840695-1767440441:73: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  now = datetime.utcnow().replace(second=0, microsecond=0)


Inserted 12 readings at 2026-10-26 15:50:05.327547


sensor_id,machine_type,reading_time,axis,peak_accel_g,vibration_rms_mm_s,temp_c
COMP-001,Compressor,2026-10-26T15:50:05.327Z,X,0.2925,0.3249,71.19
COMP-001,Compressor,2026-10-26T15:50:05.327Z,Y,0.3518,0.3148,68.53
COMP-001,Compressor,2026-10-26T15:50:05.327Z,Z,0.347,0.3525,70.98
COMP-002,Compressor,2026-10-26T15:50:05.327Z,X,0.3505,0.5756,74.89
COMP-002,Compressor,2026-10-26T15:50:05.327Z,Y,0.3853,0.5833,75.43
COMP-002,Compressor,2026-10-26T15:50:05.327Z,Z,0.4568,0.5911,74.5
MOTOR-001,Motor,2026-10-26T15:50:05.327Z,X,0.2241,0.2442,65.73
MOTOR-001,Motor,2026-10-26T15:50:05.327Z,Y,0.2567,0.2809,63.99
MOTOR-001,Motor,2026-10-26T15:50:05.327Z,Z,0.2846,0.2969,66.49
MOTOR-002,Motor,2026-10-26T15:50:05.327Z,X,0.3079,0.5011,69.2
